
# Bolometric Synchrotron Emissivity and Equipartition

The **bolometric synchrotron emissivity** is the total radiated power per unit
volume integrated over all frequencies. For a power-law electron distribution

\begin{align}N(\gamma) = N_0 \gamma^{-p}\end{align}

in a magnetic field $B$, the total emissivity follows from integrating
the single-electron synchrotron power

\begin{align}P(\gamma) =
    \frac{4}{3}\sigma_T c U_B \gamma^2,\end{align}

where $U_B = B^2/(8\pi)$ is the magnetic energy density.

The bolometric emissivity therefore becomes

\begin{align}j_{\rm bol}
    =
    \frac{4}{3}\sigma_T c \frac{B^2}{8\pi}
    \int_{\gamma_{\min}}^{\gamma_{\max}}
    N(\gamma)\gamma^2 d\gamma.\end{align}

For a fixed electron normalization $N_0$ this yields the scaling

\begin{align}j_{\rm bol} \propto N_0 B^2.\end{align}

## Equipartition

In many astrophysical systems the magnetic field and electron population are
tied to the post-shock thermal energy density $u_{\rm therm}$ through
equipartition fractions

\begin{align}u_B = \epsilon_B u_{\rm therm},
    \qquad
    u_e = \epsilon_E u_{\rm therm}.\end{align}

This implies

\begin{align}B = \sqrt{8\pi\,\epsilon_B\,u_{\rm therm}},
    \qquad
    N_0 \propto u_e \propto u_{\rm therm}.\end{align}

Since $u_{\rm therm} \propto B^2$, the electron normalization obeys

\begin{align}N_0 \propto B^2.\end{align}

Combining this with the intrinsic synchrotron scaling
$j_{\rm bol} \propto N_0 B^2$ yields

\begin{align}j_{\rm bol} \propto B^4.\end{align}

This scaling is independent of the power-law index $p$.

## Relevant API References

- :func:`~radiation.synchrotron.microphysics.compute_bol_emissivity`
- :func:`~radiation.synchrotron.microphysics.compute_bol_emissivity_BPL`
- :func:`~radiation.synchrotron.microphysics.compute_bol_emissivity_from_thermal_energy_density`
- :func:`~radiation.synchrotron.microphysics.compute_equipartition_magnetic_field`
- :func:`~radiation.synchrotron.microphysics.compute_PL_norm_from_magnetic_field`


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from astropy import units as u

from triceratops.radiation.synchrotron import (
    compute_bol_emissivity,
    compute_bol_emissivity_BPL,
    compute_bol_emissivity_from_thermal_energy_density,
    compute_equipartition_magnetic_field,
    compute_PL_norm_from_magnetic_field,
)
from triceratops.utils.plot_utils import set_plot_style

## Section 1: Bolometric Emissivity vs Magnetic Field

At fixed electron normalization $N_0$, the bolometric synchrotron
emissivity scales as

\begin{align}j_{\rm bol} \propto B^2.\end{align}

However, if the electron normalization is determined by equipartition,
then $N_0 \propto B^2$. The resulting emissivity scaling becomes

\begin{align}j_{\rm bol} \propto B^4.\end{align}

This scaling is independent of the power-law index $p$.  Below we
compute the emissivity for several values of $p$ to illustrate
that the slope is unchanged.



In [ ]:
set_plot_style()

B_arr = np.geomspace(0.01, 10.0, 200) * u.G

p_values = [2.0, 2.5, 3.0, 3.5]

gamma_min = 100
gamma_max = 1e6

epsilon_E = 0.1
epsilon_B = 0.1

fig, ax = plt.subplots(figsize=(8, 5))

for p in p_values:
    N0 = compute_PL_norm_from_magnetic_field(
        B=B_arr,
        p=p,
        epsilon_B=epsilon_B,
        epsilon_E=epsilon_E,
        gamma_min=gamma_min,
        gamma_max=gamma_max,
        mode="gamma",
    )

    j = compute_bol_emissivity(
        B=B_arr,
        N0=N0,
        p=p,
        gamma_min=gamma_min,
        gamma_max=gamma_max,
    )

    ax.loglog(
        B_arr.to_value(u.G),
        j.to_value(u.erg / u.s / u.cm**3),
        lw=2,
        label=rf"$p={p}$",
    )

# reference B^4 guide
B_ref = 0.1
j_ref = compute_bol_emissivity(
    B=B_ref * u.G,
    N0=compute_PL_norm_from_magnetic_field(
        B=B_ref * u.G,
        p=2.5,
        epsilon_B=epsilon_B,
        epsilon_E=epsilon_E,
        gamma_min=gamma_min,
        gamma_max=gamma_max,
        mode="gamma",
    ),
    p=2.5,
    gamma_min=gamma_min,
    gamma_max=gamma_max,
)

guide = j_ref.to_value(u.erg / u.s / u.cm**3) * (B_arr.value / B_ref) ** 4

ax.loglog(B_arr.value, guide, "k--", lw=1, label=r"$\propto B^4$")

ax.set_xlabel(r"Magnetic field $B$ [G]")
ax.set_ylabel(r"$j_{\rm bol}$ [erg s$^{-1}$ cm$^{-3}$]")
ax.set_title("Bolometric Synchrotron Emissivity vs Magnetic Field")
ax.legend()
ax.grid(True, which="both", ls="--", alpha=0.3)

plt.tight_layout()
plt.show()

## Section 2: Power-Law vs Broken Power-Law Emissivity

Radiative cooling modifies the electron distribution. A cooled population
is described by a broken power-law

\begin{align}N(\gamma) \propto
    \begin{cases}
        \gamma^{-p}, & \gamma < \gamma_c \\
        \gamma^{-(p+1)}, & \gamma > \gamma_c
    \end{cases}\end{align}

The steepening above $\gamma_c$ reduces the contribution of
high-energy electrons to the bolometric emissivity.  The ratio

\begin{align}j_{\rm BPL}/j_{\rm PL}\end{align}

therefore measures the suppression caused by cooling.



In [ ]:
B_demo = 0.5 * u.G
p = 2.5

gamma_c = np.geomspace(200, 5e5, 200)

N0 = compute_PL_norm_from_magnetic_field(
    B=B_demo,
    p=p,
    epsilon_B=epsilon_B,
    epsilon_E=epsilon_E,
    gamma_min=gamma_min,
    gamma_max=gamma_max,
    mode="gamma",
)

j_pl = compute_bol_emissivity(
    B=B_demo,
    N0=N0,
    p=p,
    gamma_min=gamma_min,
    gamma_max=gamma_max,
)

j_bpl = compute_bol_emissivity_BPL(
    B=B_demo,
    N0=N0,
    a1=p,
    a2=p + 1,
    gamma_b=gamma_c,
    gamma_min=gamma_min,
    gamma_max=gamma_max,
)

fig, axes = plt.subplots(2, 1, figsize=(8, 7), sharex=True)

axes[0].loglog(
    gamma_c,
    j_bpl.to_value(u.erg / u.s / u.cm**3),
    lw=2,
    label="Cooled BPL",
)

axes[0].axhline(
    j_pl.to_value(u.erg / u.s / u.cm**3),
    ls="--",
    color="k",
    label="Uncooled PL",
)

axes[0].set_ylabel(r"$j_{\rm bol}$")
axes[0].set_title("Cooling Suppression of Bolometric Emissivity")
axes[0].legend()
axes[0].grid(True, which="both", ls="--", alpha=0.3)

axes[1].semilogx(
    gamma_c,
    (j_bpl / j_pl).value,
    lw=2,
)

axes[1].axhline(1, ls="--", color="k")
axes[1].set_xlabel(r"Cooling Lorentz factor $\gamma_c$")
axes[1].set_ylabel(r"$j_{\rm BPL}/j_{\rm PL}$")
axes[1].grid(True, which="both", ls="--", alpha=0.3)

plt.tight_layout()
plt.show()

## Section 3: Equipartition Magnetic Field

The equipartition magnetic field follows directly from

\begin{align}u_B = \frac{B^2}{8\pi} = \epsilon_B u_{\rm therm}.\end{align}

Therefore

\begin{align}B_{\rm eq} = \sqrt{8\pi\,\epsilon_B\,u_{\rm therm}}.\end{align}

This produces the characteristic scaling

\begin{align}B_{\rm eq} \propto u_{\rm therm}^{1/2}.\end{align}



In [ ]:
u_therm = np.geomspace(1e-4, 1e6, 200) * u.erg / u.cm**3

eps_vals = [0.01, 0.1, 0.33, 1]

fig, ax = plt.subplots(figsize=(8, 5))

for eps in eps_vals:
    B_eq = compute_equipartition_magnetic_field(
        u_therm=u_therm,
        epsilon_B=eps,
    )

    ax.loglog(
        u_therm.to_value(u.erg / u.cm**3),
        B_eq.to_value(u.G),
        lw=2,
        label=rf"$\epsilon_B={eps}$",
    )

# guide slope
B_ref = compute_equipartition_magnetic_field(
    u_therm=1 * u.erg / u.cm**3,
    epsilon_B=0.1,
)

guide = B_ref.value * (u_therm.value) ** 0.5

ax.loglog(u_therm.value, guide, "k--", label=r"$\propto u_{\rm therm}^{1/2}$")

ax.set_xlabel(r"$u_{\rm therm}$ [erg cm$^{-3}$]")
ax.set_ylabel(r"$B_{\rm eq}$ [G]")
ax.set_title("Equipartition Magnetic Field Scaling")
ax.legend()
ax.grid(True, which="both", ls="--", alpha=0.3)

plt.tight_layout()
plt.show()